# Решение тестового задания

Источник данных: `Копия Данные для тестового задания.xlsx`.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

FILE = "Копия Данные для тестового задания.xlsx"

aud = pd.read_excel(FILE, sheet_name="Данные об аудитории")
ab = pd.read_excel(FILE, sheet_name="Данные АБ тестов")
listers = pd.read_excel(FILE, sheet_name="Листеры")

aud["date"] = pd.to_datetime(aud["date"])
print(aud.shape, ab.shape, listers.shape)
print("Листы:", ["Данные об аудитории", "Данные АБ тестов", "Листеры"])

## 1. MAU

In [ ]:
mau = aud["user_id"].nunique()
print("MAU =", mau)

## 2. DAU

In [ ]:
daily_dau = aud.groupby(aud["date"].dt.date)["user_id"].nunique()
print("Средний DAU =", daily_dau.mean())
display(daily_dau.to_frame("DAU"))

## 3. Retention D1

In [ ]:
nov1 = set(aud.loc[aud["date"].dt.date == pd.Timestamp("2023-11-01").date(), "user_id"])
nov2 = set(aud.loc[aud["date"].dt.date == pd.Timestamp("2023-11-02").date(), "user_id"])
retention_d1 = len(nov1 & nov2) / len(nov1)
print("1 ноября:", len(nov1))
print("Вернулись 2 ноября:", len(nov1 & nov2))
print("Retention D1 =", retention_d1)

## 4. Retention-кривые

Продукт 1 удерживает пользователей лучше: кривая снижается медленнее. У продукта 2 retention быстро падает до 0%.

## 5. Пользовательская конверсия в просмотр объявления

In [ ]:
user_views = aud.groupby("user_id")["view_adverts"].sum()
view_users = (user_views > 0).sum()
conversion = view_users / mau
print("Пользователей с просмотром:", view_users)
print("Конверсия:", conversion, "=", conversion*100, "%")

## 6. Среднее число просмотренных объявлений на пользователя

In [ ]:
avg_views = user_views.mean()
print("Среднее:", avg_views)

## 7. NPS

In [ ]:
nps = (1200/2000 - 500/2000) * 100
print("NPS =", nps, "%")

## 8. A/B-тесты ARPU

In [ ]:
results = []
for exp, g in ab.groupby("experiment_num"):
    control = g.loc[g["experiment_group"] == "control", "revenue"].astype(float)
    test = g.loc[g["experiment_group"] == "test", "revenue"].astype(float)
    t_stat, p_value = stats.ttest_ind(control, test, equal_var=False)
    results.append({
        "experiment": int(exp),
        "control_ARPU": control.mean(),
        "test_ARPU": test.mean(),
        "p_value": p_value,
        "significant_0.05": p_value < 0.05
    })
ab_results = pd.DataFrame(results)
display(ab_results)

**Интерпретация:** эксперимент 1 — различие незначимо; эксперимент 2 — test статистически значимо хуже по ARPU; эксперимент 3 — test выше по ARPU, но при α=0,05 различие статистически незначимо. Поэтому окончательно выбирать test в эксперименте 3 только по этим данным не следует.

## 9. Средний доход на пользователя среди листеров

In [ ]:
avg_income = listers["revenue"].sum() / listers["user_id"].nunique()
print("Средний доход на пользователя =", avg_income)

## 10. Медиана возраста

In [ ]:
median_age = listers.groupby("user_id")["age"].first().median()
print("Медиана возраста =", median_age)

## 11–17. Теория

11. Ящик с усами (box plot).

12. №3 — бимодальное распределение.

13. №3 — наибольшая дисперсия.

14. №1 и №3 — Scatter Plot и Correlation Heatmap.

15. №2 — корректная интерпретация p-value = 0,05.

16. t-тест.

17. Квартили делят данные на четыре равные части.

## 18. A/B-тест конверсии

In [ ]:
nA, xA = 100_047_501, 1003
nB, xB = 100_001_055, 1099

pA = xA / nA
pB = xB / nB
p_pool = (xA + xB) / (nA + nB)
se = np.sqrt(p_pool * (1-p_pool) * (1/nA + 1/nB))
z = (pB - pA) / se
p_value = 2 * stats.norm.sf(abs(z))

print("Conversion A =", pA)
print("Conversion B =", pB)
print("z =", z)
print("p-value =", p_value)
print("Recommendation:", "B" if p_value < 0.05 else "No statistically significant winner")

**Итог:** p-value ≈ 0,0353 < 0,05, поэтому различие статистически значимо и вариант B рекомендуется. Относительный uplift ≈ 9,62%.